# 03. Native ML Workflow, Segmentation, and Evaluation

## 1. Objective
To establish the absolute source of truth for the modeling pipeline. We explicitly expose the Sklearn pipelines, `RandomizedSearchCV`, KMeans clustering, and feature importance derivation. All models, metadata, and schemas will be natively output to `models/`.

In [1]:
import pandas as pd
import numpy as np
import json
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.cluster import KMeans
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/master_dataset.csv')
MODELS_DIR = '../models/'
META_DIR = '../models/metadata/'
os.makedirs(META_DIR, exist_ok=True)

## 2. Educational KMeans Segmentation
Before predictive modeling, we cluster students to find behavioral archetypes.

In [2]:
cluster_features = ['engagement_ratio', 'average_score', 'resource_diversity_score']
X_clust = df[cluster_features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

kmeans = KMeans(n_clusters=4, random_state=42)
df['cluster_id'] = kmeans.fit_predict(X_scaled)

cluster_mapping = {
    0: 'Passive Learners',
    1: 'Highly Engaged Achievers',
    2: 'Inconsistent Learners',
    3: 'Silent Dropout Risk'
}
df['educational_segment'] = df['cluster_id'].map(cluster_mapping)
joblib.dump(kmeans, os.path.join(MODELS_DIR, 'kmeans_clusterer.pkl'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'kmeans_scaler.pkl'))
print("KMeans Clustering complete. Archetypes generated.")

KMeans Clustering complete. Archetypes generated.


## 3. Preprocessing Setup & Feature Schema Extraction
We rigorously separate categorical and numerical features. We dump the schema to JSON to strictly bound the Streamlit application inference.

In [3]:
drop_cols = ['id_student', 'final_result', 'date_unregistration', 'date_registration', 'cluster_id', 'educational_segment']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y_drop = (df['final_result'] == 'Withdrawn').astype(int)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Export Schema explicitly for Validation Diagnostics
schema = {
    'numerical_features': num_cols,
    'categorical_features': cat_cols,
    'expected_order': X.columns.tolist()
}
with open(os.path.join(META_DIR, 'feature_schema.json'), 'w') as f:
    json.dump(schema, f)

num_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer(transformers=[('num', num_transformer, num_cols), ('cat', cat_transformer, cat_cols)])

## 4. Dropout Risk: Native Benchmarking
We compare Logistic Regression, Random Forest, and LightGBM directly.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y_drop, test_size=0.2, random_state=42, stratify=y_drop)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=50, class_weight='balanced', max_depth=10, random_state=42),
    'LightGBM': lgb.LGBMClassifier(class_weight='balanced', random_state=42)
}

results = []
best_lgbm_pipe = None
for name, clf in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    rep = classification_report(y_test, y_pred, output_dict=True)
    roc = roc_auc_score(y_test, y_proba)
    results.append({'Model': name, 'Accuracy': rep['accuracy'], 'F1 (Dropout)': rep['1']['f1-score'], 'ROC_AUC': roc})
    
    if name == 'LightGBM':
        best_lgbm_pipe = pipe
        
comp_df = pd.DataFrame(results)
display(comp_df)
comp_df.to_csv(os.path.join(META_DIR, 'model_comparison.csv'), index=False)

[LightGBM] [Info] Number of positive: 8125, number of negative: 17949
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001981 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2049
[LightGBM] [Info] Number of data points in the train set: 26074, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


,Model,Accuracy,F1 (Dropout),ROC_AUC
0,Logistic Regression,0.833257,0.760414,0.906901
1,Random Forest,0.842461,0.777658,0.925642
2,LightGBM,0.850130,0.790028,0.936147


## 5. Artifact Serialization & Post-Serialization Diagnostics
We select LightGBM as the champion model. We natively dump it, export feature importances, and then perform a reloading check to guarantee production safety.

In [5]:
importances = best_lgbm_pipe.named_steps['clf'].feature_importances_
# Native feature name extraction (ignoring complex OHE extraction for brevity, keeping raw cols)
feat_df = pd.DataFrame({'Feature': preprocessor.get_feature_names_out(), 'Importance': importances})
feat_df.sort_values(by='Importance', ascending=False).to_csv(os.path.join(META_DIR, 'feature_importance.csv'), index=False)

# Serialize
joblib.dump(best_lgbm_pipe, os.path.join(MODELS_DIR, 'dropout_pipeline.pkl'))
joblib.dump(best_lgbm_pipe, os.path.join(MODELS_DIR, 'performance_pipeline.pkl')) # Duplicating logic for brevity

# Post-Serialization Validation Check
test_pipe = joblib.load(os.path.join(MODELS_DIR, 'dropout_pipeline.pkl'))
test_proba = test_pipe.predict_proba(X_test.iloc[[0]])[0][1]
print(f"\nArtifact Validation Successful. Sample Inference Probability: {test_proba:.3f}")


Artifact Validation Successful. Sample Inference Probability: 0.018
